# UPC RAG — Discovery Notebook (Fase 1)

Mapea el sitio público UPC haciendo BFS de profundidad limitada **sin descargar nada**. Salida: 4 CSVs con todas las URLs encontradas, clasificadas por sección, con score de relevancia. Sirve para revisar qué hay antes de correr el scraper completo (Fase 2).

- **Profundidad**: 2 niveles desde cada semilla (configurable)
- **Delay**: 2-4s aleatorio entre requests
- **Domain lock**: solo `*.upc.edu.pe`
- **Sin Playwright** en discovery — si una página tiene poco texto se anota y se decide en Fase 2 si vale la pena renderizarla con browser headless

## 1 · Instalación de dependencias

In [1]:
%pip install -q requests beautifulsoup4 tqdm pandas lxml ipywidgets openpyxl

Note: you may need to restart the kernel to use updated packages.


## 2 · Imports y configuración global

Todos los parámetros editables viven aquí. Cambia las semillas, profundidad o keywords según lo que quieras explorar.

In [2]:
import re
import time
import random
from collections import deque, Counter
from datetime import datetime
from urllib.parse import urlparse, urljoin, urldefrag
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import pandas as pd

# === Semillas ===
SEED_URLS = [
    "https://pregrado.upc.edu.pe/facultad-de-ingenieria/ingenieria-de-sistemas-de-informacion/",
    "https://pregrado.upc.edu.pe/facultad-de-ingenieria/",
    "https://www.upc.edu.pe/admision/becas-y-financiamiento/becas-internas-alumnos/",
    "https://www.upc.edu.pe/admision/becas-y-financiamiento/",
    "https://explora.upc.edu.pe/",
    "https://www.upc.edu.pe/estudia-con-nosotros/matricula/",
    "https://www.upc.edu.pe/estudia-con-nosotros/calendario-academico/",
]

# === Crawl ===
MAX_DEPTH = 2          # niveles desde cada semilla (semilla = depth 0)
DELAY_MIN = 2.0
DELAY_MAX = 4.0
TIMEOUT = 15           # segundos
MAX_PAGES = 500        # cap defensivo para evitar runaways

USER_AGENT = (
    "Mozilla/5.0 (compatible; UPC-RAG-Discovery/1.0; "
    "+https://github.com/upc-thesis/chatbot)"
)

ALLOWED_DOMAIN_SUFFIX = ".upc.edu.pe"

SKIP_PATTERNS = [
    "login", "signin", "campus-net", "intranet", "blackboard",
    "canvas", "aula-virtual", "facebook.com", "twitter.com", "x.com",
    "instagram", "youtube", "linkedin", "whatsapp",
]

# Esquemas no-HTTP que ignoramos al extraer links
SKIP_SCHEMES = ("mailto:", "javascript:", "tel:")

KEYWORDS = [
    "matrícula", "matricula", "horario", "cronograma", "calendario",
    "malla", "currícula", "curricula", "reglamento", "beca",
    "financiamiento", "pago", "pensión", "pension", "requisito",
    "ciclo", "crédito", "credito", "egreso", "ingresante",
    "grado", "titulación", "titulacion",
]

# === Output ===
OUTPUT_DIR = Path("./discovery_output")
OUTPUT_DIR.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

print(f"Semillas: {len(SEED_URLS)}")
print(f"Profundidad máxima: {MAX_DEPTH}")
print(f"Cap de páginas: {MAX_PAGES}")
print(f"Output: {OUTPUT_DIR.resolve()}")

Semillas: 7
Profundidad máxima: 2
Cap de páginas: 500
Output: /Users/renzolenes/Desktop/Proyectos/chatbot-upc/scrapping/discovery_output


## 3 · Funciones auxiliares

Clasificación de URL → sección, validación de dominio, normalización, extracción de texto y links.

In [3]:
def normalize_url(url: str) -> str:
    """Quita fragmento (#), normaliza espacios."""
    url, _ = urldefrag(url.strip())
    return url


def is_allowed_domain(url: str) -> bool:
    """True si el host termina en .upc.edu.pe o es upc.edu.pe."""
    host = urlparse(url).hostname or ""
    return host == "upc.edu.pe" or host.endswith(ALLOWED_DOMAIN_SUFFIX)


def should_skip(url: str) -> str | None:
    """Devuelve el patrón que matchea o None si no se debe saltar."""
    if url.startswith(SKIP_SCHEMES):
        return "non-http-scheme"
    low = url.lower()
    for pat in SKIP_PATTERNS:
        if pat in low:
            return pat
    return None


def classify_section(url: str) -> str:
    """Asigna subcarpeta lógica por la URL.
    pregrado.upc.edu.pe/* → pregrado
    */becas* → becas
    */matricula* → matricula
    */reglamento* → reglamentos
    resto → otros
    """
    parsed = urlparse(url)
    host = parsed.hostname or ""
    path = parsed.path.lower()
    if host.startswith("pregrado."):
        return "pregrado"
    if "beca" in path or "financiamiento" in path:
        return "becas"
    if "matricula" in path or "matrícula" in path:
        return "matricula"
    if "reglamento" in path:
        return "reglamentos"
    return "otros"


def extract_visible_text(soup: BeautifulSoup) -> str:
    """Texto del contenido principal, sin nav/header/footer/script/style."""
    work = BeautifulSoup(str(soup), "lxml")
    for tag in work(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
        tag.decompose()
    # Elementos comunes que en WP/UPC son sidebar/menus
    for sel in [".sidebar", ".menu", ".breadcrumb", "#wpadminbar"]:
        for el in work.select(sel):
            el.decompose()
    text = work.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text)


def count_keywords(text: str) -> tuple[int, list[str]]:
    """Cuenta keywords (case-insensitive). Retorna (total_hits, lista_unica)."""
    low = text.lower()
    hits = []
    for kw in KEYWORDS:
        if kw in low:
            hits.append(kw)
    return len(hits), hits


def extract_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    """Lista de URLs absolutas, normalizadas y deduplicadas."""
    seen = set()
    out = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href:
            continue
        absolute = urljoin(base_url, href)
        absolute = normalize_url(absolute)
        if absolute and absolute not in seen:
            seen.add(absolute)
            out.append(absolute)
    return out


print("helpers cargados ✓")

helpers cargados ✓


## 4 · `fetch_simple(url)` — request + parsing

En esta fase **no usamos Playwright**. Si una página parece tener poco contenido (probable SPA), la marcamos para revisar en Fase 2.

In [4]:
def fetch_simple(url: str) -> dict:
    """Fetch y parse simple. Retorna dict con: ok, status, html, soup, error."""
    result = {
        "ok": False,
        "status": None,
        "html": None,
        "soup": None,
        "error": None,
    }
    try:
        resp = session.get(url, timeout=TIMEOUT, allow_redirects=True)
        result["status"] = resp.status_code
        if resp.status_code != 200:
            result["error"] = f"HTTP {resp.status_code}"
            return result
        ctype = resp.headers.get("Content-Type", "").lower()
        if "html" not in ctype:
            result["error"] = f"non-html: {ctype}"
            return result
        result["html"] = resp.text
        result["soup"] = BeautifulSoup(resp.text, "lxml")
        result["ok"] = True
    except requests.Timeout:
        result["error"] = "timeout"
    except requests.RequestException as e:
        result["error"] = f"request: {type(e).__name__}"
    except Exception as e:
        result["error"] = f"unexpected: {type(e).__name__}: {e}"
    return result


# Smoke test rápido
_test = fetch_simple(SEED_URLS[0])
print("OK" if _test["ok"] else "FAIL", "|", _test["status"], "|", _test["error"])

OK | 200 | None


## 5 · `discover()` — BFS depth-limited con CSV streaming

Escribe cada fila a `/tmp/upc_discovery/*.csv` apenas se genera. Si el kernel se cuelga, los datos ya están en disco. Usa `/tmp` para esquivar la sincronización de iCloud Drive del Desktop.

Retorna 4 DataFrames idénticos al contenido de los CSVs:

1. **pages** — URLs HTML visitadas (con métricas: palabras, keywords hit, links)
2. **pdfs** — URLs `.pdf` encontradas (NO descargadas, solo listadas)
3. **skipped** — URLs ignoradas y por qué
4. **errors** — URLs con error HTTP/timeout


In [5]:
import csv

CHECKPOINT_DIR = Path("/tmp/upc_discovery")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Schemas — columnas de cada CSV (orden fijo)
PAGES_COLS   = ["url", "depth", "parent", "section", "title", "num_words",
                "num_keywords", "keywords_hit", "num_links_total",
                "num_links_internal", "num_pdf_links", "likely_spa", "relevant"]
PDFS_COLS    = ["pdf_url", "section", "depth", "found_in_page"]
SKIPPED_COLS = ["url", "depth", "parent", "reason"]
ERRORS_COLS  = ["url", "depth", "parent", "section", "error", "status"]


class IncrementalCSV:
    """Wrapper que escribe filas a CSV apenas se agregan. Auto-flush para
    que siempre haya algo en disco si el kernel se muere."""
    def __init__(self, path, columns):
        self.path = path
        self.columns = columns
        self.file = open(path, "w", newline="", encoding="utf-8")
        self.writer = csv.DictWriter(self.file, fieldnames=columns)
        self.writer.writeheader()
        self.file.flush()
        self.rows = []  # también en memoria para construir el DF al final

    def append(self, row):
        self.writer.writerow(row)
        self.file.flush()  # CRÍTICO: vuelca al disco inmediatamente
        self.rows.append(row)

    def close(self):
        self.file.close()


def discover(seed_urls, max_depth=MAX_DEPTH, max_pages=MAX_PAGES):
    """BFS depth-limited. Escribe cada fila al CSV en /tmp apenas se genera."""
    visited = set()

    pages_csv   = IncrementalCSV(CHECKPOINT_DIR / "discovery_pages.csv",   PAGES_COLS)
    pdfs_csv    = IncrementalCSV(CHECKPOINT_DIR / "discovery_pdfs.csv",    PDFS_COLS)
    skipped_csv = IncrementalCSV(CHECKPOINT_DIR / "discovery_skipped.csv", SKIPPED_COLS)
    errors_csv  = IncrementalCSV(CHECKPOINT_DIR / "discovery_errors.csv",  ERRORS_COLS)

    queue: deque = deque()
    for seed in seed_urls:
        queue.append((normalize_url(seed), 0, None))

    pbar = tqdm(total=max_pages, desc="discovering")
    pages_count = 0

    try:
        while queue and pages_count < max_pages:
            url, depth, parent = queue.popleft()
            if url in visited:
                continue
            visited.add(url)

            skip_reason = should_skip(url)
            if skip_reason:
                skipped_csv.append({"url": url, "depth": depth, "parent": parent, "reason": skip_reason})
                continue
            if not is_allowed_domain(url):
                skipped_csv.append({"url": url, "depth": depth, "parent": parent, "reason": "external-domain"})
                continue

            section = classify_section(url)

            if url.lower().endswith(".pdf"):
                pdfs_csv.append({"pdf_url": url, "section": section, "depth": depth, "found_in_page": parent})
                pbar.update(1)
                pages_count += 1
                continue

            result = fetch_simple(url)
            if not result["ok"]:
                errors_csv.append({"url": url, "depth": depth, "parent": parent, "section": section,
                                   "error": result["error"], "status": result["status"]})
                pbar.update(1)
                pages_count += 1
                time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
                continue

            soup = result["soup"]
            text = extract_visible_text(soup)
            num_words = len(text.split())
            num_kw, kw_hits = count_keywords(text)
            title = (soup.title.string.strip() if soup.title and soup.title.string else "")

            links = extract_links(soup, url)
            internal_links = [l for l in links if is_allowed_domain(l)]
            pdf_links = [l for l in links if l.lower().endswith(".pdf")]

            pages_csv.append({
                "url": url, "depth": depth, "parent": parent, "section": section,
                "title": title[:160], "num_words": num_words,
                "num_keywords": num_kw, "keywords_hit": ",".join(kw_hits),
                "num_links_total": len(links), "num_links_internal": len(internal_links),
                "num_pdf_links": len(pdf_links),
                "likely_spa": num_words < 200,
                "relevant": num_words >= 300 and num_kw >= 1,
            })

            for pdf in pdf_links:
                pdfs_csv.append({"pdf_url": pdf, "section": classify_section(pdf),
                                 "depth": depth + 1, "found_in_page": url})

            if depth < max_depth:
                for link in internal_links:
                    if link.lower().endswith(".pdf"):
                        continue
                    if link not in visited:
                        queue.append((link, depth + 1, url))

            pbar.update(1)
            pages_count += 1
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    finally:
        # Cierre garantizado aunque haya KeyboardInterrupt o crash
        pages_csv.close()
        pdfs_csv.close()
        skipped_csv.close()
        errors_csv.close()
        pbar.close()
        print(f"\n✓ CSVs guardados en {CHECKPOINT_DIR}/")

    # DataFrames desde la memoria (idénticos a los CSVs)
    pages_df   = pd.DataFrame(pages_csv.rows)
    pdfs_df    = pd.DataFrame(pdfs_csv.rows).drop_duplicates(subset=["pdf_url"]).reset_index(drop=True)
    skipped_df = pd.DataFrame(skipped_csv.rows)
    errors_df  = pd.DataFrame(errors_csv.rows)

    return pages_df, pdfs_df, skipped_df, errors_df


print("discover() incremental listo — escribe a", CHECKPOINT_DIR)


discover() incremental listo — escribe a /tmp/upc_discovery


## 6 · Ejecutar discovery

Con `MAX_DEPTH=2` y delays 2-4s, esperar ~5-15 min según cuántos enlaces internos haya. La barra de progreso muestra hasta `MAX_PAGES`.

In [6]:
started_at = datetime.now()
pages_df, pdfs_df, skipped_df, errors_df = discover(SEED_URLS, MAX_DEPTH, MAX_PAGES)
elapsed = (datetime.now() - started_at).total_seconds()
print(f"\nTerminado en {elapsed:.0f}s")
print(f"  pages   : {len(pages_df)}")
print(f"  pdfs    : {len(pdfs_df)}")
print(f"  skipped : {len(skipped_df)}")
print(f"  errors  : {len(errors_df)}")

discovering:   0%|          | 0/500 [00:00<?, ?it/s]


✓ CSVs guardados en /tmp/upc_discovery/

Terminado en 1741s
  pages   : 475
  pdfs    : 719
  skipped : 3
  errors  : 25


## 7 · Verificar CSVs y exportar Excel (opcional)

Los CSVs ya se escribieron incrementalmente durante el discovery. Esta celda solo confirma que están en disco y opcionalmente arma un Excel multi-hoja para revisión humana.


In [7]:
# 1) Confirmar archivos en disco
import os
for name in ["pages", "pdfs", "skipped", "errors"]:
    p = CHECKPOINT_DIR / f"discovery_{name}.csv"
    if p.exists():
        size_kb = p.stat().st_size / 1024
        with open(p) as f:
            num_rows = sum(1 for _ in f) - 1  # menos el header
        print(f"  {p.name:30s}  {num_rows:>5d} filas  {size_kb:>6.1f} KB")
    else:
        print(f"  {p.name}  ✗ no existe")

# 2) (opcional) Excel multi-hoja para inspección humana
EXCEL_PATH = CHECKPOINT_DIR / "discovery.xlsx"
with pd.ExcelWriter(EXCEL_PATH) as writer:
    pages_df.to_excel(writer, sheet_name="pages", index=False)
    pdfs_df.to_excel(writer, sheet_name="pdfs", index=False)
    skipped_df.to_excel(writer, sheet_name="skipped", index=False)
    errors_df.to_excel(writer, sheet_name="errors", index=False)
print(f"\n✓ Excel: {EXCEL_PATH}")


  discovery_pages.csv               475 filas   112.1 KB
  discovery_pdfs.csv                806 filas   143.6 KB
  discovery_skipped.csv               3 filas     0.3 KB
  discovery_errors.csv               25 filas     3.3 KB

✓ Excel: /tmp/upc_discovery/discovery.xlsx


## 8 · Análisis — qué semillas dieron más juego, qué secciones tienen más material

In [8]:
# Distribución por sección
print("=== Páginas HTML por sección ===")
print(pages_df.groupby("section").agg(
    paginas=("url", "count"),
    relevantes=("relevant", "sum"),
    likely_spa=("likely_spa", "sum"),
    palabras_prom=("num_words", "mean"),
    keywords_prom=("num_keywords", "mean"),
).round(1))

print("\n=== PDFs encontrados por sección ===")
print(pdfs_df.groupby("section").size().rename("pdfs"))

print("\n=== Distribución por profundidad ===")
print(pages_df.groupby("depth").size().rename("pages"))

print("\n=== Páginas marcadas likely_spa (poco texto, candidatas a Playwright) ===")
print(pages_df[pages_df["likely_spa"]].groupby("section").size().rename("likely_spa"))

=== Páginas HTML por sección ===
             paginas  relevantes  likely_spa  palabras_prom  keywords_prom
section                                                                   
becas             14          10           2          605.6            5.4
matricula          1           0           1           71.0            1.0
otros            280         178          79         1208.7            3.3
pregrado         136         104          20         1419.6            3.8
reglamentos       44           2          42          108.5            1.4

=== PDFs encontrados por sección ===
section
becas         17
matricula      3
otros        683
pregrado      16
Name: pdfs, dtype: int64

=== Distribución por profundidad ===
depth
0      5
1     77
2    393
Name: pages, dtype: int64

=== Páginas marcadas likely_spa (poco texto, candidatas a Playwright) ===
section
becas           2
matricula       1
otros          79
pregrado       20
reglamentos    42
Name: likely_spa, dtype: int64


In [9]:
# Top 20 páginas más relevantes (más keywords + más palabras)
print("=== Top 20 páginas más relevantes ===")
top_pages = (
    pages_df[pages_df["relevant"]]
    .sort_values(["num_keywords", "num_words"], ascending=False)
    .head(20)
    [["section", "depth", "num_keywords", "num_words", "title", "url"]]
)
with pd.option_context("display.max_colwidth", 60):
    print(top_pages.to_string(index=False))

=== Top 20 páginas más relevantes ===
section  depth  num_keywords  num_words                                                                                                   title                                                                                url
  otros      1            17      13483                                                                                             Explora UPC                                               https://explora.upc.edu.pe/ayuda-upc
  otros      1            16      25463                                                                                             Explora UPC                                        https://explora.upc.edu.pe/alumno-postgrado
  otros      1            14      38374                                                                                             Explora UPC                                        https://explora.upc.edu.pe/79939-biblioteca
  otros      1            14       9959               

In [10]:
# Patrones de skip más comunes — útil para refinar SKIP_PATTERNS
print("=== Razones de skip más comunes ===")
if not skipped_df.empty:
    print(skipped_df["reason"].value_counts().head(15))
else:
    print("(ninguna URL skippeada)")

print("\n=== Errores ===")
if not errors_df.empty:
    print(errors_df["error"].value_counts())
    print("\nMuestra de URLs con error:")
    print(errors_df[["url", "error", "status"]].head(10).to_string(index=False))
else:
    print("(sin errores)")

=== Razones de skip más comunes ===
reason
login           2
aula-virtual    1
Name: count, dtype: int64

=== Errores ===
error
HTTP 404                    21
HTTP 403                     3
request: ConnectionError     1
Name: count, dtype: int64

Muestra de URLs con error:
                                                              url                    error  status
           https://www.upc.edu.pe/estudia-con-nosotros/matricula/                 HTTP 404   404.0
https://www.upc.edu.pe/estudia-con-nosotros/calendario-academico/                 HTTP 404   404.0
                    https://epe.upc.edu.pe/admision-epe/convenios                 HTTP 403   403.0
                 https://postgrado.upc.edu.pe/executive-education                 HTTP 404   404.0
           https://noticias.upc.edu.pe/cdn-cgi/l/email-protection                 HTTP 404   404.0
       https://www3.upc.edu.pe/html/0/1/admisionEPE/index-epe.htm request: ConnectionError     NaN
          https://postgrado.upc.

In [11]:
# Productividad por semilla — ¿cuál descubrió más material útil?
print("=== Material descubierto por semilla ===")
rows = []
for seed in SEED_URLS:
    norm_seed = normalize_url(seed)
    descendientes = pages_df[
        (pages_df["url"] == norm_seed) | (pages_df["parent"].str.startswith(norm_seed.split("://")[1].split("/")[0], na=False))
    ]
    pdfs_descend = pdfs_df[
        pdfs_df["found_in_page"].str.contains(urlparse(seed).hostname, na=False)
    ]
    rows.append({
        "semilla": seed,
        "section": classify_section(seed),
        "paginas_descubiertas": len(descendientes),
        "relevantes": int(descendientes["relevant"].sum()) if not descendientes.empty else 0,
        "pdfs_unicos": pdfs_descend["pdf_url"].nunique(),
    })
with pd.option_context("display.max_colwidth", 80):
    print(pd.DataFrame(rows).to_string(index=False))

=== Material descubierto por semilla ===
                                                                                  semilla   section  paginas_descubiertas  relevantes  pdfs_unicos
https://pregrado.upc.edu.pe/facultad-de-ingenieria/ingenieria-de-sistemas-de-informacion/  pregrado                     1           1          145
                                      https://pregrado.upc.edu.pe/facultad-de-ingenieria/  pregrado                     1           1          145
           https://www.upc.edu.pe/admision/becas-y-financiamiento/becas-internas-alumnos/     becas                     1           1          248
                                  https://www.upc.edu.pe/admision/becas-y-financiamiento/     becas                     1           1          248
                                                              https://explora.upc.edu.pe/     otros                     1           1           78
                                   https://www.upc.edu.pe/estudia-con-nosotro

In [12]:
# Keywords que más aparecen — pista para ajustar la lista
print("=== Keywords más frecuentes en páginas relevantes ===")
all_hits = []
for s in pages_df["keywords_hit"].dropna():
    if s:
        all_hits.extend([k.strip() for k in s.split(",") if k.strip()])
kw_count = Counter(all_hits)
for kw, n in kw_count.most_common():
    print(f"  {kw:18s} {n}")

=== Keywords más frecuentes en páginas relevantes ===
  grado              310
  ciclo              177
  curricula          139
  malla              132
  pago               126
  ingresante         97
  reglamento         86
  horario            85
  requisito          74
  financiamiento     64
  beca               62
  cronograma         54
  matricula          33
  pension            29
  calendario         25
  matrícula          23
  crédito            23
  titulación         14
  pensión            8
  egreso             8


## 9 · Próximos pasos

Antes de correr la Fase 2 (scraper completo con Playwright + descarga):

1. **Revisar `discovery_pages.csv`**: ¿hay páginas relevantes que no esperabas? ¿páginas claramente irrelevantes (eventos, noticias) que deberíamos sumar a `SKIP_PATTERNS`?
2. **Revisar `discovery_pdfs.csv`**: ¿el conteo es razonable? Si son cientos de PDFs antes de descargar, verificar que sean docs académicos y no banners/folletos.
3. **Revisar `likely_spa=True`**: estas páginas tienen poco texto en HTML estático — probable contenido renderizado con JS. Decidir si en Fase 2 vale la pena pagar el costo de Playwright para todas o solo para algunas secciones.
4. **Refinar keywords**: si una keyword sale 0 veces, considerar quitarla. Si una keyword común no está, agregarla.
5. **Refinar semillas**: si una semilla descubrió 0 páginas relevantes, posiblemente cambió la URL o el contenido — verificar manualmente.

Una vez ajustada la config, pasar al notebook **`02_scraper.ipynb`** que sí descarga los PDFs y guarda el texto extraído.